# LLM Orchestration: Multi-Provider Routing, Sampling Controls, and System Prompts

**Objective:** Benchmark LLM responses across multiple infrastructure providers (Groq, Google Gemini, and local Ollama). This analysis evaluates the behavioral impact of system prompts, tracks cross-provider token discrepancies, and tests how stochastic sampling parameters (temperature, max_tokens, stop sequences) impact deterministic intent routing.

In [1]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from providers_utils import ask, classify_ticket, gemini_client, groq_client, list_gemini_models, ollama_client, pick_gemini_model
from tickets_data import ALL_TICKETS, BORDERLINE_TICKETS, CLEAR_TICKETS

groq = groq_client()
GROQ_MODEL = os.getenv("MODEL")

try:
    gemini = gemini_client()
    # Gemini's Flash model name changes often - verify instead of trusting a name written down months ago.
    flash_models = list_gemini_models()
    GEMINI_MODEL = os.getenv("GEMINI_MODEL") or pick_gemini_model(flash_models)
    print("Gemini models available:", flash_models)
    print("using:", GEMINI_MODEL)
except Exception as e:
    gemini, GEMINI_MODEL = None, None
    print("Gemini not configured yet:", e)

try:
    ollama = ollama_client()
    OLLAMA_MODEL = "llama3.2:3b"
except Exception as e:
    ollama = None
    print("Ollama not available:", e)


Gemini models available: ['gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite', 'gemini-3.1-flash-lite-preview', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-flash-latest', 'gemini-flash-lite-latest']
using: gemini-3.8-flash


## 1. System Prompt Impact on Output Formatting
The system message dictates the model's persona and constraints. Here we measure the output differential between a strictly prompted request and a default state.

In [2]:
system = "You are a terse assistant for a retail bank. Answer in one short sentence."
user = "What is an EMI?"

with_system = ask(groq, GROQ_MODEL, system, user)
without_system = ask(groq, GROQ_MODEL, "", user)

print("WITH system message:\n ", with_system["text"])
print("\nWITHOUT a system message:\n ", without_system["text"])


WITH system message:
  EMI is the fixed monthly payment you make to repay a loan.

WITHOUT a system message:
  **


**Key Finding — Prompt Enforcement:** Using a system prompt successfully forces the model into a single-sentence, professional output, whereas omitting the system prompt causes the model to default to a verbose, generic paragraph. Dynamic provider switching is managed strictly at the environment variable level (e.g., swapping `GROQ_API_KEY` for `GEMINI_API_KEY`), completely isolating the application code from provider-specific logic.

## 2. Cross-Provider Token Usage Benchmarks
We send the identical system prompt and user query across three separate architectures (Groq API, Gemini API, and local Ollama) to compare raw text generation and token counting discrepancies.

In [ ]:
prompt = "In one sentence, what is the difference between a workflow and an agent?"
results = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("gemini", gemini, GEMINI_MODEL), ("ollama", ollama, "llama3.2:3b" if ollama else None)]:
    if client is None or model is None:
        print(f"--- {label}: skipped (not configured)\n"); continue
    try:
        r = ask(client, model, "You are a concise AI engineering tutor.", prompt)
        results[label] = r
        print(f"--- {label} ({model})")
        print(" ", r["text"].strip())
        print(f"   tokens: prompt={r['prompt_tokens']} completion={r['completion_tokens']} total={r['total_tokens']}\n")
    except Exception as e:
        print(f"--- {label}: error ({type(e).__name__}: {str(e)[:100]})\n")

--- groq (openai/gpt-oss-20b)
  A workflow is a predefined, orchestrated sequence of tasks, while an agent is an autonomous entity that perceives, reasons, and acts to achieve goals.
   tokens: prompt=97 completion=112 total=209



**Key Finding — Token Count vs. Cost Efficiency:** The local Ollama deployment utilized the fewest total tokens (91) for the exact same prompt, but lower token counts do not automatically equate to better or cheaper performance. While cloud providers like Groq and Gemini charge strictly per token, self-hosted models incur continuous, fixed infrastructure and compute costs regardless of usage. Ultimately, model routing decisions must be calculated using a matrix of response correctness, Time to First Token (TTFT), and Tokens Per Second (TPS).

## 3. Deterministic Controls: Max Tokens and Stop Sequences
Hard limits must be enforced on generated text to prevent runaway API costs or broken UI layouts.
* `max_tokens`: Hard cap on length.
* `stop`: Kills the generation sequence the moment a specific string (like a newline character `\n`) appears.

In [ ]:
# Demonstrating truncation via max_tokens
short = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=15)
enough = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=150)
print("max_tokens=15:\n ", short["text"], "\n")
print("max_tokens=150:\n ", enough["text"][:300], "\n")

# Demonstrating hard stops via stop sequences
r = ask(groq, GROQ_MODEL, "You are a helpful assistant. Answer in a short paragraph.",
        "List three benefits of retrieval-augmented generation.", max_tokens=200, stop=["\n"])
print("Stopped at first newline:\n", r["text"])

## 4. Stochastic Variance (Temperature Sweep) on Intent Routing
Temperature controls randomness. Here we evaluate the risk of stochastic sampling on structured business logic by running a borderline support ticket through a classification router at multiple temperature thresholds (0.0 to 1.0).

In [ ]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]
RUNS_PER_TEMPERATURE = 3

if groq is None:
    print("skipped (no Groq client)")
else:
    for tid, text in BORDERLINE_TICKETS:
        print(f"\n=== {tid} ===\n{text}\n")
        for temp in TEMPERATURES:
            labels = []
            for _ in range(RUNS_PER_TEMPERATURE):
                result = classify_ticket(groq, GROQ_MODEL, text, temperature=temp)
                labels.append(result["label"] if result["valid"] else f"INVALID:{result['label']!r}")
            stable = len(set(labels)) == 1
            print(f"  temperature={temp:<4} {'stable' if stable else 'DRIFTED':8} {labels}")


**Key Finding — Production Temperature Thresholds:** When classifying ambiguous data (borderline tickets), labels began to drift significantly at `temperature=0.7`, causing stochastic sampling to generate inconsistent classifications and invalid empty strings (`INVALID:''`)[cite: 3]. Conversely, unambiguous inputs remained stable across all temperatures because the probability of the top predicted token was overwhelmingly dominant[cite: 3]. To prevent downstream pipeline failures, any LLM deployed for deterministic production tasks—such as intent classification or JSON schema extraction—must strictly be locked to `temperature=0.0`[cite: 3].